# MySteth 2.0 — Component 1: Signal Quality Assessment (SQA)

**Task.** Predict whether a CirCor recording is **Good** (heart sounds usable) or **Poor**, from audio only, so later components can gate on it.

**Labels.** `circor_main_labels.csv`: all 3,163 recordings from 942 patients, 2,154 Good and 1,009 Poor. All are gold labels. They were either labelled by ear, or accepted from the weak labeller after a blind Cohen's κ spot-check (see *Why manual labels*).

**Model.** Five frozen pretrained audio encoders (PANNs embedding, PANNs AudioSet class scores, AST, CLAP, HeAR). Each is pooled over 10 s windows and joined with 7 hand-crafted features. A logistic regression is trained on each, and the five probabilities are averaged (late fusion).

**Protocol.**
- Patient-grouped 5-fold cross-validation. Recordings of the same child never sit on both sides of a split, and that includes a child who appears under two patient IDs (`Additional ID`).
- The regularisation strength `C` and the Good/Poor threshold are chosen *inside* each training fold (4-fold inner CV), so the test fold is never used for any choice.
- Metrics: **F1, Accuracy, Weighted accuracy (balanced), AUC**, with patient-level bootstrap 95% CIs. Models are evaluated on all 3,163 labels.

**Kaggle setup.** GPU on, Internet on. Add an `HF_TOKEN` secret for HeAR, unless its embeddings are already cached. Attach:
1. the CirCor v2 dataset,
2. your dataset with the new `circor_main_labels.csv`,
3. `sqa-helpers` (`recording_features_full_v2.csv`),
4. *(optional)* a dataset with the cached `emb_*.npy` + `emb_ids.csv` from the earlier runs. This skips about 25 min of embedding.

## Why manual labels

The first SQA version used weak supervision. 261 recordings were labelled by hand, and a Snorkel LabelModel with 8 labelling functions over hand-crafted acoustic features labelled the other 2,902. On the gold set it reached 0.828 cross-validated accuracy. Three findings made those labels unsuitable as ground truth:

1. **They failed an independent check.** On 167 newly hand-labelled recordings that were never used to build the labeller, weak-vs-human agreement was **κ = 0.423**, below the κ ≥ 0.60 reliability bar applied to the human labels. The errors went one way:
   - Weak-Good was almost always right (37 of 39).
   - Weak-Poor was right only 82 of 128 times.
   - 448 recordings (15% of the pool) received no labelling-function vote and fell to *Poor* by default. In the check, 13 of 14 of these were in fact Good.
   - Moving the probability threshold raised agreement only to κ = 0.63.
2. **They were circular for this task.** The labelling functions are built on the same hand-crafted features an SQA model learns from. Scoring an SQA model against them measures agreement with those features, not with what a listener hears.
3. **They mixed in the diagnosis.** One labelling function used the murmur class as a prior (*Unknown* → Poor). Quality labels built this way partly encode the diagnostic label that later MySteth components predict, and RQ1 relates SQA to exactly those components.

**What was done instead: targeted manual labelling.**
- **Weak-Good:** accepted only after a blind spot-check, 60 weak-Good recordings plus 40 previously hand-labelled Poor controls. **κ = 0.832**, and 57 of the 60 were heard as Good.
- **No-vote recordings:** same design. **κ = 0.726**, and 55 of 60 were heard as Good, so they were accepted as Good.
- **All weak-Poor recordings:** labelled by ear in batches. Each batch has a blind re-check, and a batch counts only if intra-rater κ ≥ 0.60. Across the 8 re-checks, κ ranged from 0.613 to 0.795.
- **Model-assisted labelling:** tested on a held-out batch and rejected, because agreement was κ 0.58–0.665.

The result is 3,163 gold labels: each was either heard, or accepted through a passed κ check.

In [1]:
!pip install -q panns_inference

In [2]:
import time, glob, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from scipy.signal import resample_poly
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, balanced_accuracy_score
from lightgbm import LGBMClassifier
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 240)
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

# ---- EDIT THESE PATHS to match your Kaggle inputs ----
CIRCOR_ROOT   = Path("/kaggle/input/datasets/bjoernjostein/the-circor-digiscope-phonocardiogram-dataset-v2")
AUDIO_DIR     = CIRCOR_ROOT / "training_data" / "training_data"
LABELS_PATH   = Path("/kaggle/input/datasets/tusharrbb/labels-mannual/circor_main_labels.csv")
FEATURES_PATH = Path("/kaggle/input/datasets/tusharrbb/sqa-helpers/recording_features_full_v2.csv")
EMB_CACHE_DIR = Path("/kaggle/input/datasets/tusharrbb/embeddings-sqa")
OUT_DIR = Path("/kaggle/working"); OUT_DIR.mkdir(parents=True, exist_ok=True)

SRC_SR = 4000
WIN_SEC, HOP_SEC, MIN_LAST_SEC = 10.0, 5.0, 3.0          # pooling windows for PANNs / AST / CLAP
HANDCRAFTED = ["rms_energy", "spectral_centroid", "spectral_bandwidth", "zcr",
               "clip_ratio", "silence_ratio", "periodicity_strength"]
ENCODERS = ["panns", "panns_probs", "ast", "clap", "hear"]  # 'panns_probs' comes from the same PANNs forward pass
N_OUTER, N_INNER = 5, 4
C_GRID = [0.001, 0.01, 0.1]
TH_GRID = np.round(np.arange(0.05, 0.951, 0.025), 3)
N_BOOT = 1000

device: cuda Tesla T4


## 1. Labels, patient groups and folds

In [3]:
def wav_id(p):
    s = str(p).replace("\\", "/").split("/")[-1]
    return s[:-4] if s.endswith(".wav") else s

labels = pd.read_csv(LABELS_PATH)
feats = pd.read_csv(FEATURES_PATH)
labels["wid"] = labels.wav_path.map(wav_id)
feats["wid"] = feats.wav_path.map(wav_id)

assert set(labels.quality) == {"Good", "Poor"}, labels.quality.value_counts()
assert labels.wid.is_unique and set(labels.wid) == set(feats.wid), "labels and features must cover the same recordings"

data = (labels.merge(feats[["wid"] + HANDCRAFTED], on="wid")
              .sort_values("wid").reset_index(drop=True))     # fixed order -> reproducible folds
data["y"] = (data.quality == "Good").astype(int)
assert data[HANDCRAFTED].notna().all().all()

# Children who took part in both CirCor campaigns have two patient IDs, linked via 'Additional ID'.
# Merge such pairs into one group so the same child can't be in both train and test.
meta_path = next(iter(sorted(glob.glob(str(CIRCOR_ROOT / "**" / "training_data.csv"), recursive=True))), None)
assert meta_path, "training_data.csv (CirCor patient metadata) not found under CIRCOR_ROOT"
meta = pd.read_csv(meta_path)
parent = {}
def find(a):
    parent.setdefault(a, a)
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
for pid, add in zip(meta["Patient ID"], meta["Additional ID"]):
    find(int(pid))
    if pd.notna(add):
        parent[find(int(pid))] = find(int(add))
data["group"] = data.patient_id.astype(int).map(find)
print(f"metadata: {meta_path}")
print(f"recordings {len(data)} | patients {data.patient_id.nunique()} | groups after linking repeat visits {data.group.nunique()}")

sgkf = StratifiedGroupKFold(n_splits=N_OUTER, shuffle=True, random_state=SEED)
data["fold"] = -1
for f, (_, te) in enumerate(sgkf.split(data, data.y, groups=data.group)):
    data.loc[te, "fold"] = f
assert data.groupby("group").fold.nunique().max() == 1

OUTER = [(np.where(data.fold != f)[0], np.where(data.fold == f)[0]) for f in range(N_OUTER)]
INNER = []                                    # inner splits are shared by every model -> fusion stays aligned
for tr, _ in OUTER:
    d = data.iloc[tr]
    ik = StratifiedGroupKFold(n_splits=N_INNER, shuffle=True, random_state=SEED + 1)
    INNER.append([(tr[a], tr[b]) for a, b in ik.split(d, d.y, groups=d.group)])

Y = data.y.values

print("\nlabels:", data.quality.value_counts().to_dict())
print("\nfolds:")
print(data.groupby("fold").agg(recordings=("y", "size"), groups=("group", "nunique"),
                               good_rate=("y", "mean")).round(3))

metadata: /kaggle/input/datasets/bjoernjostein/the-circor-digiscope-phonocardiogram-dataset-v2/training_data.csv
recordings 3163 | patients 942 | groups after linking repeat visits 872

labels: {'Good': 2154, 'Poor': 1009}

folds:
      recordings  groups  good_rate
fold                               
0            628     174      0.686
1            650     175      0.718
2            630     175      0.678
3            608     167      0.643
4            647     181      0.677


## 2. Audio embeddings

Each encoder sees the 4 kHz audio resampled to its own rate:

| key | model | rate | per-window vector |
|---|---|---|---|
| `panns` | PANNs CNN14 (AudioSet) | 32 kHz | 2048-d embedding |
| `panns_probs` | same forward pass | 32 kHz | 527 AudioSet class scores (*Baby cry*, *Speech*, *Breathing*, …) |
| `ast` | AST (`MIT/ast-finetuned-audioset-10-10-0.4593`) | 16 kHz | 768-d |
| `clap` | CLAP (`laion/clap-htsat-unfused`) | 48 kHz | 512-d |
| `hear` | HeAR (`google/hear`, health acoustics) | 16 kHz | 512-d per 2 s clip |

PANNs, AST and CLAP use 10 s windows with a 5 s hop, and HeAR uses 2 s clips with a 1 s hop. All are pooled per recording as **mean ‖ max** over windows, because the max keeps a single crying burst visible. Embeddings are cached in `/kaggle/working`, or loaded from `EMB_CACHE_DIR`.

In [4]:
ALL_IDS = data.wid.tolist()                      # sorted, = the 3,163 labelled recordings


def load_audio(wid):
    x, sr = sf.read(AUDIO_DIR / f"{wid}.wav", dtype="float32", always_2d=False)
    if x.ndim > 1:
        x = x.mean(axis=1)
    assert sr == SRC_SR, (wid, sr)
    return x


def resample(x, target_sr):
    return resample_poly(x, target_sr // SRC_SR, 1).astype(np.float32)   # integer factors from 4 kHz


def windows(x, sr):
    # 10 s windows, 5 s hop; keep a final partial window if >= 3 s; short recordings -> one window
    w, h = int(WIN_SEC * sr), int(HOP_SEC * sr)
    if len(x) <= w:
        return [x]
    out = [x[s:s + w] for s in range(0, len(x) - w + 1, h)]
    last = (len(x) - w) // h * h + h
    if len(x) - last >= MIN_LAST_SEC * sr:
        out.append(x[last:])
    return out


def pool(mat):
    return np.concatenate([mat.mean(axis=0), mat.max(axis=0)])


def _as_tensor(out):                              # transformers 4.x returns a tensor, 5.x may return a ModelOutput
    if isinstance(out, torch.Tensor):
        return out
    for k in ("pooler_output", "audio_embeds", "last_hidden_state"):
        v = getattr(out, k, None)
        if v is not None:
            return v if v.dim() == 2 else v.mean(dim=1)
    return out[0]


PANNS_CKPT = Path.home() / "panns_data" / "Cnn14_mAP=0.431.pth"
PANNS_URL = "https://zenodo.org/records/3987831/files/Cnn14_mAP=0.431.pth?download=1"

def fetch_panns_checkpoint():                     # one progress bar instead of wget's log spam
    import urllib.request
    if PANNS_CKPT.exists() and PANNS_CKPT.stat().st_size >= 3e8:
        return
    PANNS_CKPT.parent.mkdir(parents=True, exist_ok=True)
    tmp = PANNS_CKPT.with_suffix(".part")
    with urllib.request.urlopen(PANNS_URL) as r, open(tmp, "wb") as f, \
         tqdm(total=int(r.headers.get("Content-Length", 0)), unit="B", unit_scale=True, desc="PANNs checkpoint") as bar:
        while chunk := r.read(1 << 20):
            f.write(chunk); bar.update(len(chunk))
    tmp.replace(PANNS_CKPT)


class PANNsEncoder:
    sr, outputs = 32000, ["panns", "panns_probs"]
    def __init__(self):
        fetch_panns_checkpoint()
        from panns_inference import AudioTagging
        self.at = AudioTagging(checkpoint_path=None, device=DEVICE)
    @torch.no_grad()
    def __call__(self, x):
        e, p = [], []
        for w in windows(x, self.sr):
            clip, emb = self.at.inference(w[None, :])
            p.append(clip[0]); e.append(emb[0])
        return {"panns": pool(np.stack(e)), "panns_probs": pool(np.stack(p))}


class ASTEncoder:
    sr, outputs, NAME = 16000, ["ast"], "MIT/ast-finetuned-audioset-10-10-0.4593"
    def __init__(self):
        from transformers import ASTFeatureExtractor, ASTModel
        self.fe = ASTFeatureExtractor.from_pretrained(self.NAME)
        self.m = ASTModel.from_pretrained(self.NAME).to(DEVICE).eval()
    @torch.no_grad()
    def __call__(self, x):
        inp = self.fe(windows(x, self.sr), sampling_rate=self.sr, return_tensors="pt").to(DEVICE)
        return {"ast": pool(self.m(**inp).pooler_output.float().cpu().numpy())}


class CLAPEncoder:
    sr, outputs, NAME = 48000, ["clap"], "laion/clap-htsat-unfused"
    def __init__(self):
        from transformers import ClapFeatureExtractor, ClapModel
        self.fe = ClapFeatureExtractor.from_pretrained(self.NAME)
        self.m = ClapModel.from_pretrained(self.NAME).to(DEVICE).eval()
    @torch.no_grad()
    def __call__(self, x):
        # windows are <= 10 s, so 'rand_trunc' never crops; shorter ones are repeat-padded
        inp = self.fe(windows(x, self.sr), sampling_rate=self.sr, return_tensors="pt",
                      truncation="rand_trunc", padding="repeatpad").to(DEVICE)
        return {"clap": pool(_as_tensor(self.m.get_audio_features(**inp)).float().cpu().numpy())}


class HeAREncoder:
    sr, outputs, CLIP, HOP = 16000, ["hear"], 32000, 16000      # 2 s clips, 1 s hop
    def __init__(self):
        import tensorflow as tf
        for g in tf.config.list_physical_devices("GPU"):
            tf.config.experimental.set_memory_growth(g, True)   # don't let TF take the GPU from PyTorch
        from huggingface_hub import snapshot_download, login
        from kaggle_secrets import UserSecretsClient
        login(token=UserSecretsClient().get_secret("HF_TOKEN"))
        self.tf = tf
        self.fn = tf.saved_model.load(snapshot_download("google/hear")).signatures["serving_default"]
    def clips(self, x):
        if len(x) <= self.CLIP:
            return [np.pad(x, (0, self.CLIP - len(x)))]
        starts = list(range(0, len(x) - self.CLIP + 1, self.HOP))
        if len(x) - (starts[-1] + self.CLIP) >= self.sr:         # final partial clip if >= 1 s
            starts.append(len(x) - self.CLIP)
        return [x[s:s + self.CLIP] for s in starts]
    def __call__(self, x):
        c = np.stack(self.clips(x)).astype(np.float32)
        return {"hear": pool(self.fn(x=self.tf.constant(c))["output_0"].numpy())}


ENCODER_CLASSES = [PANNsEncoder, ASTEncoder, CLAPEncoder, HeAREncoder]


def cached(key):
    for d in [OUT_DIR] + ([Path(EMB_CACHE_DIR)] if EMB_CACHE_DIR else []):
        e, i = d / f"emb_{key}.npy", d / "emb_ids.csv"
        if e.exists() and i.exists():
            ids = pd.read_csv(i).wid.tolist()
            arr = np.load(e)
            assert set(ALL_IDS) <= set(ids), f"cached {key} misses recordings"
            pos = {w: k for k, w in enumerate(ids)}
            return arr[[pos[w] for w in ALL_IDS]]
    return None


EMB = {}
for cls in ENCODER_CLASSES:
    wanted = [k for k in cls.outputs if k in ENCODERS]
    if not wanted:
        continue
    for k in wanted:
        arr = cached(k)
        if arr is not None:
            EMB[k] = arr
    if all(k in EMB for k in wanted):
        print(f"{cls.__name__}: loaded from cache")
        continue
    t0, enc, rows = time.time(), cls(), {k: [] for k in cls.outputs}
    for wid in tqdm(ALL_IDS, desc=cls.__name__, unit="rec", mininterval=2):
        for k, v in enc(resample(load_audio(wid), enc.sr)).items():
            rows[k].append(v)
    for k, v in rows.items():
        EMB[k] = np.stack(v).astype(np.float32)
        np.save(OUT_DIR / f"emb_{k}.npy", EMB[k])
    pd.Series(ALL_IDS, name="wid").to_csv(OUT_DIR / "emb_ids.csv", index=False)
    print(f"{cls.__name__}: computed in {time.time() - t0:.0f}s")
    del enc
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

EMB = {k: EMB[k] for k in ENCODERS}
print({k: v.shape for k, v in EMB.items()})

PANNsEncoder: loaded from cache
ASTEncoder: loaded from cache
CLAPEncoder: loaded from cache
HeAREncoder: loaded from cache
{'panns': (3163, 4096), 'panns_probs': (3163, 1054), 'ast': (3163, 1536), 'clap': (3163, 1024), 'hear': (3163, 1024)}


## 3. Models and nested cross-validation

| model | input | classifier | role |
|---|---|---|---|
| `handcrafted LogReg` | 7 hand-crafted features | logistic regression | simple baseline |
| `handcrafted LightGBM` | 7 hand-crafted features | LightGBM, same configuration as the earlier Component 1 model | the previous SQA model, now scored against human labels |
| `<encoder>+hc` ×5 | one embedding + 7 hand-crafted | logistic regression (`C` tuned) | single-encoder probes |
| **`SQA fusion (5)`** | — | mean of the five probes' probabilities | **final model** |

For each of the 5 outer folds:
1. Inner 4-fold CV on the training part. Each candidate `C` produces inner out-of-fold probabilities, and the `C` with the best AUC is kept.
2. The threshold is the one that maximises weighted (balanced) accuracy on those inner out-of-fold probabilities. For fusion, the five members' inner probabilities are averaged first.
3. The model is refitted on the whole training part and scores the test fold.

In [5]:
FINAL_TRAIN = "all labels"
TRAIN_SETS = {"all labels": np.ones(len(data), bool)}
HC = data[HANDCRAFTED].values.astype(np.float32)

SPECS = {"handcrafted LogReg": ("logreg", None), "handcrafted LightGBM": ("lgbm", None)}
SPECS.update({f"{k}+hc": ("logreg", k) for k in ENCODERS})
FUSION_MEMBERS = [f"{k}+hc" for k in ENCODERS]
FUSION = "SQA fusion (5)"


def features(enc):
    return HC if enc is None else np.hstack([EMB[enc], HC]).astype(np.float32)


def build(kind, C=None):
    if kind == "logreg":
        return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000, class_weight="balanced"))
    return LGBMClassifier(n_estimators=300, num_leaves=15, min_child_samples=10, learning_rate=0.05,
                          class_weight="balanced", random_state=SEED, verbose=-1)


def best_threshold(y, p):
    return float(TH_GRID[np.argmax([balanced_accuracy_score(y, (p >= t).astype(int)) for t in TH_GRID])])


def nested(name, train_set):
    kind, enc = SPECS[name]
    X, use = features(enc), TRAIN_SETS[train_set]
    oof, inner_p, chosen = np.full(len(data), np.nan), [], []
    for f, (tr, te) in enumerate(OUTER):
        best = None
        for C in (C_GRID if kind == "logreg" else [None]):
            p = np.full(len(data), np.nan)
            for itr, iva in INNER[f]:
                fit_rows = itr[use[itr]]
                p[iva] = build(kind, C).fit(X[fit_rows], Y[fit_rows]).predict_proba(X[iva])[:, 1]
            auc = roc_auc_score(Y[tr], p[tr])
            if best is None or auc > best[0]:
                best = (auc, C, p)
        fit_rows = tr[use[tr]]
        oof[te] = build(kind, best[1]).fit(X[fit_rows], Y[fit_rows]).predict_proba(X[te])[:, 1]
        inner_p.append(best[2]); chosen.append(best[1])
    return {"oof": oof, "inner": inner_p, "C": chosen}


def with_thresholds(r):
    th = np.full(len(data), np.nan)
    for f, (tr, te) in enumerate(OUTER):
        th[te] = best_threshold(Y[tr], r["inner"][f][tr])
    r["th"] = th
    return r


RUNS = {}
for ts in TRAIN_SETS:
    for name in SPECS:
        t0 = time.time()
        RUNS[(name, ts)] = with_thresholds(nested(name, ts))
        print(f"{name:22s} C per fold {RUNS[(name, ts)]['C']}  ({time.time() - t0:.0f}s)")
    members = [RUNS[(m, ts)] for m in FUSION_MEMBERS]
    RUNS[(FUSION, ts)] = with_thresholds({
        "oof": np.mean([m["oof"] for m in members], axis=0),
        "inner": [np.mean([m["inner"][f] for m in members], axis=0) for f in range(N_OUTER)],
        "C": None})
assert all(np.isfinite(r["oof"]).all() for r in RUNS.values())

handcrafted LogReg     C per fold [0.1, 0.1, 0.1, 0.1, 0.1]  (1s)
handcrafted LightGBM   C per fold [None, None, None, None, None]  (4s)
panns+hc               C per fold [0.001, 0.001, 0.001, 0.001, 0.001]  (68s)
panns_probs+hc         C per fold [0.01, 0.01, 0.01, 0.01, 0.01]  (42s)
ast+hc                 C per fold [0.01, 0.01, 0.001, 0.01, 0.01]  (54s)
clap+hc                C per fold [0.01, 0.01, 0.01, 0.01, 0.01]  (55s)
hear+hc                C per fold [0.01, 0.01, 0.01, 0.001, 0.01]  (50s)


## 4. Results

The threshold differs per outer fold, because each is chosen inside that fold's training data. AUC doesn't use the threshold. 95% CIs come from a patient-level bootstrap: whole children are resampled, not single recordings.

In [6]:
GROUP_IDX = {}
for g, idx in data.groupby("group").indices.items():
    GROUP_IDX[g] = idx
ALL = np.ones(len(data), bool)
SUBSETS = {"All labels": ALL}


def scores(y, p, th):
    pr = (p >= th).astype(int)
    return {"F1_macro": f1_score(y, pr, average="macro"), "F1_Good": f1_score(y, pr, pos_label=1),
            "F1_Poor": f1_score(y, pr, pos_label=0), "Accuracy": accuracy_score(y, pr),
            "Weighted_acc": balanced_accuracy_score(y, pr), "AUC": roc_auc_score(y, p)}


def boot_indices(mask, n=N_BOOT, seed=SEED):
    groups = [idx[mask[idx]] for idx in GROUP_IDX.values()]
    groups = [g for g in groups if len(g)]
    rng = np.random.default_rng(seed)
    for _ in range(n):
        yield np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])


CI_KEYS = ["F1_macro", "Accuracy", "Weighted_acc", "AUC"]
rows = []
for (name, ts), r in RUNS.items():
    for sub, mask in SUBSETS.items():
        idx = np.where(mask)[0]
        row = {"train": ts, "subset": sub, "model": name, "n": len(idx), **scores(Y[idx], r["oof"][idx], r["th"][idx])}
        bs = np.array([[scores(Y[b], r["oof"][b], r["th"][b])[k] for k in CI_KEYS] for b in boot_indices(mask, n=300)])
        for j, k in enumerate(CI_KEYS):
            row[f"{k}_CI"] = f"[{np.percentile(bs[:, j], 2.5):.3f}, {np.percentile(bs[:, j], 97.5):.3f}]"
        rows.append(row)
RESULTS = pd.DataFrame(rows)
RESULTS.to_csv(OUT_DIR / "sqa_results.csv", index=False)

ORDER = list(SPECS) + [FUSION]
for sub in SUBSETS:
    t = RESULTS[(RESULTS.train == FINAL_TRAIN) & (RESULTS.subset == sub)].set_index("model").loc[ORDER]
    print(f"\n=== {sub}  (trained on: {FINAL_TRAIN}, n = {t.n.iloc[0]}) ===")
    print(t.drop(columns=["train", "subset", "n"]).round(3).to_string())
print("\nfinal-model thresholds per fold:", sorted(set(np.round(RUNS[(FUSION, FINAL_TRAIN)]['th'], 3))))


=== All labels  (trained on: all labels, n = 3163) ===
                      F1_macro  F1_Good  F1_Poor  Accuracy  Weighted_acc    AUC     F1_macro_CI     Accuracy_CI Weighted_acc_CI          AUC_CI
model                                                                                                                                          
handcrafted LogReg       0.813    0.864    0.762     0.827         0.838  0.900  [0.799, 0.826]  [0.813, 0.839]  [0.825, 0.852]  [0.889, 0.910]
handcrafted LightGBM     0.800    0.849    0.750     0.812         0.832  0.907  [0.784, 0.815]  [0.796, 0.826]  [0.818, 0.845]  [0.896, 0.917]
panns+hc                 0.801    0.856    0.746     0.816         0.824  0.908  [0.785, 0.814]  [0.802, 0.829]  [0.810, 0.837]  [0.897, 0.918]
panns_probs+hc           0.805    0.856    0.755     0.819         0.834  0.910  [0.791, 0.819]  [0.803, 0.832]  [0.822, 0.847]  [0.900, 0.920]
ast+hc                   0.798    0.852    0.744     0.812         0.824  0.908 

## 5. Comparisons

Each row is a **paired** patient-level bootstrap: the same resampled children are scored by both models, so the CI is for the *difference*. If a CI excludes 0, the difference is unlikely to be bootstrap noise.

The final model is compared against each baseline and each single encoder, on all labels.

In [7]:
def paired(a, b, mask, label):
    idx = np.where(mask)[0]
    sa, sb = scores(Y[idx], a["oof"][idx], a["th"][idx]), scores(Y[idx], b["oof"][idx], b["th"][idx])
    d = []
    for bi in boot_indices(mask):
        xa, xb = scores(Y[bi], a["oof"][bi], a["th"][bi]), scores(Y[bi], b["oof"][bi], b["th"][bi])
        d.append([xa[k] - xb[k] for k in CI_KEYS])
    d = np.array(d)
    row = {"comparison": label}
    for j, k in enumerate(CI_KEYS):
        lo, hi = np.percentile(d[:, j], [2.5, 97.5])
        row[f"d_{k}"] = round(sa[k] - sb[k], 3)
        row[f"d_{k}_CI"] = f"[{lo:+.3f}, {hi:+.3f}]"
    return row


final = RUNS[(FUSION, FINAL_TRAIN)]
cmp = [paired(final, RUNS[(m, FINAL_TRAIN)], ALL, f"{FUSION} − {m}") for m in SPECS]
CMP = pd.DataFrame(cmp)
print("=== Final model vs each alternative — all labels ===")
print(CMP.to_string(index=False))
CMP.to_csv(OUT_DIR / "sqa_comparisons.csv", index=False)

=== Final model vs each alternative — all labels ===
                           comparison  d_F1_macro    d_F1_macro_CI  d_Accuracy    d_Accuracy_CI  d_Weighted_acc d_Weighted_acc_CI  d_AUC         d_AUC_CI
  SQA fusion (5) − handcrafted LogReg       0.011 [-0.002, +0.025]       0.010 [-0.003, +0.023]           0.012  [-0.000, +0.027]  0.029 [+0.022, +0.038]
SQA fusion (5) − handcrafted LightGBM       0.024 [+0.011, +0.040]       0.025 [+0.012, +0.040]           0.019  [+0.005, +0.033]  0.022 [+0.015, +0.031]
            SQA fusion (5) − panns+hc       0.023 [+0.013, +0.034]       0.021 [+0.011, +0.031]           0.027  [+0.015, +0.038]  0.022 [+0.016, +0.028]
      SQA fusion (5) − panns_probs+hc       0.019 [+0.008, +0.029]       0.018 [+0.009, +0.028]           0.016  [+0.006, +0.028]  0.019 [+0.014, +0.025]
              SQA fusion (5) − ast+hc       0.026 [+0.016, +0.037]       0.025 [+0.015, +0.035]           0.026  [+0.016, +0.037]  0.021 [+0.017, +0.027]
             SQA fusion

## 6. Final model and outputs for the other components

- **`sqa_oof_predictions.csv`**: one row per recording, with `p_good` and `pred_good` from the final model *out of fold*. Each recording is scored by a model that never saw that child. **Downstream components (RQ1, RQ6, …) should read `p_good` from this file**, so the SQA score never leaks a recording's own label into later analyses.
- **`sqa_final_model.joblib`**: the five probes refitted on all training labels, plus the threshold (median of the 5 outer-fold thresholds). This is only for scoring *new* recordings, which first need the same embeddings (§2).

In [8]:
import joblib
from collections import Counter

fold_th = [float(np.unique(final["th"][te])[0]) for _, te in OUTER]
FINAL_THRESHOLD = float(np.median(fold_th))

use = TRAIN_SETS[FINAL_TRAIN]
members = {}
for m in FUSION_MEMBERS:
    kind, enc = SPECS[m]
    C = Counter(RUNS[(m, FINAL_TRAIN)]["C"]).most_common(1)[0][0]
    members[m] = {"encoder": enc, "C": C, "model": build(kind, C).fit(features(enc)[use], Y[use])}
joblib.dump({"members": members, "threshold": FINAL_THRESHOLD, "handcrafted": HANDCRAFTED,
             "encoders": ENCODERS, "pooling": "mean||max over windows", "trained_on": FINAL_TRAIN,
             "positive_class": "Good"}, OUT_DIR / "sqa_final_model.joblib")

out = data[["wid", "patient_id", "group", "location", "murmur_class", "quality", "y", "fold"]].copy()
out["p_good"] = final["oof"]
out["threshold"] = final["th"]
out["pred_good"] = (out.p_good >= out.threshold).astype(int)
out.to_csv(OUT_DIR / "sqa_oof_predictions.csv", index=False)

print(f"fold thresholds {fold_th} -> final threshold {FINAL_THRESHOLD}")
print("final C per member:", {m: v["C"] for m, v in members.items()})
print("\npredicted Good rate (OOF):", round(out.pred_good.mean(), 3), "| true Good rate:", round(out.y.mean(), 3))
print(pd.crosstab(out.quality, out.pred_good.map({1: "pred Good", 0: "pred Poor"}), margins=True))
print("\nsaved:", sorted(p.name for p in OUT_DIR.iterdir() if p.is_file() and not p.name.startswith("emb_")))

fold thresholds [0.575, 0.55, 0.55, 0.625, 0.5] -> final threshold 0.55
final C per member: {'panns+hc': 0.001, 'panns_probs+hc': 0.01, 'ast+hc': 0.01, 'clap+hc': 0.01, 'hear+hc': 0.01}

predicted Good rate (OOF): 0.589 | true Good rate: 0.681
pred_good  pred Good  pred Poor   All
quality                              
Good            1751        403  2154
Poor             113        896  1009
All             1864       1299  3163

saved: ['__notebook__.ipynb', 'sqa_comparisons.csv', 'sqa_final_model.joblib', 'sqa_oof_predictions.csv', 'sqa_results.csv']


## What to report

- **Benchmark:** §4, *All labels*, model `SQA fusion (5)`: F1-macro, Accuracy, Weighted accuracy, AUC with their CIs.
- **Improvement over the previous SQA approach:** row `SQA fusion (5) − handcrafted LightGBM` in §5.
- **Label reliability:** your own intra-rater κ from the labelling app. Models are not scored with κ.